In [ ]:
import importlib
import detection_utils_modifié
importlib.reload(detection_utils_modifié)

from detection_utils_modifié import detect_hands, process_npz, process_npy_folder

hands = detect_hands(detection_confidence=0.7)

In [ ]:
import os
import numpy as np
import tensorflow as tf

from tensorflow.keras import Input, Model
from tensorflow.keras.layers import (
    TimeDistributed,
    GlobalAveragePooling2D,
    LSTM,
    Dense,
    BatchNormalization,
    Dropout
)
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint,
    TensorBoard
)
import matplotlib.pyplot as plt

In [ ]:
T = 20                  
size = 224         
classes_number = 69        
batch_size = 16
epochs = 15
learning_rate = 1e-4

In [ ]:
def npz_generator(npz_path, sequential):
    data = np.load(npz_path, mmap_mode="r")
    X, y = data["X"], data["y"]
    N = X.shape[0]
    for i in range(N):
        xi = X[i]  # shape = (t, H, W, C) ou (1, H, W, C)
        label = int(y[i])
        if not sequential:
            xi = np.repeat(xi, repeats=T, axis=0)
        yield xi.astype(np.float32), label

In [ ]:
def npy_generator(folder_path, sequential):
    X = np.load(os.path.join(folder_path, "X.npy"), mmap_mode="r")
    y = np.load(os.path.join(folder_path, "y.npy"), mmap_mode="r")
    N = X.shape[0]
    for i in range(N):
        xi = X[i]  # shape = (t, H, W, C) ou (1, H, W, C)
        label = int(y[i])
        if not sequential:
            xi = np.repeat(xi, repeats=T, axis=0)
        yield xi.astype(np.float32), label

In [ ]:
def load_npz_via_generator(npz_path, sequential):
    output_types = (tf.float32, tf.int32)
    output_shapes = ((T, size, size, 3), ())
    ds = tf.data.Dataset.from_generator(
        lambda: npz_generator(npz_path, sequential),
        output_types=output_types,
        output_shapes=output_shapes
    )
    return (
        ds
        .shuffle(buffer_size=1000)      
        .batch(batch_size)             
        .prefetch(tf.data.AUTOTUNE)
    )

In [ ]:
def load_npy_via_generator(folder_path, sequential):
    output_types = (tf.float32, tf.int32)
    output_shapes = ((T, size, size, 3), ())
    ds = tf.data.Dataset.from_generator(
        lambda: npy_generator(folder_path, sequential),
        output_types=output_types,
        output_shapes=output_shapes
    )
    return (
        ds
        .shuffle(buffer_size=1000)
        .batch(batch_size)
        .prefetch(tf.data.AUTOTUNE)
    )

In [ ]:
train_seq_ds = load_npz_via_generator(r"C:\Users\21654\Downloads\dataset_again\train_sequences.npz", sequential=True)
val_seq_ds   = load_npz_via_generator(r"C:\Users\21654\Downloads\dataset_again\valid_sequences.npz",   sequential=True)

train_static_ds = load_npy_via_generator(r"C:\Users\21654\Downloads\dataset_again\train_static_unzipped", sequential=False)
val_static_ds   = load_npz_via_generator(r"C:\Users\21654\Downloads\dataset_again\valid_static.npz",   sequential=False)

In [ ]:
train_ds = train_seq_ds.concatenate(train_static_ds)
val_ds   = val_seq_ds.concatenate(val_static_ds)

In [ ]:
X_batch, y_batch = next(iter(train_ds))
plt.imshow(X_batch[0,0]); plt.title(int(y_batch[0])); plt.show()

In [ ]:
inputs = Input(shape=(T, size, size, 3))
base = MobileNetV2(input_shape=(size, size, 3),
                   include_top=False, weights="imagenet")
base.trainable = False


x = TimeDistributed(base)(inputs)
x = TimeDistributed(GlobalAveragePooling2D())(x)
x = LSTM(128)(x)
x = Dense(1024, activation="relu")(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)
x = Dense(512, activation="relu")(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)
outputs = Dense(classes_number, activation="softmax")(x)

model = Model(inputs, outputs)
model.compile(optimizer=Adam(learning_rate),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy",
                       tf.keras.metrics.TopKCategoricalAccuracy(3)])
model.summary()

In [ ]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3),
    ModelCheckpoint("best_model.h5", save_best_only=True),
    TensorBoard(log_dir="logs", histogram_freq=1)
]

In [ ]:
history = model.fit(train_ds,
                    validation_data=val_ds,
                    epochs=epochs,
                    callbacks=callbacks)

In [ ]:
test_seq_ds = load_npz_via_generator(r"C:\Users\21654\Downloads\dataset_again\test_sequences.npz", sequential=True)

test_static_ds = load_npz_via_generator(r"C:\Users\21654\Downloads\dataset_again\test_static.npz", sequential=False)

test_ds = test_seq_ds.concatenate(test_static_ds)

results = model.evaluate(test_ds)
print(f"Test loss: {results[0]:.4f}")
print(f"Test accuracy: {results[1]:.4%}")
print(f"Test top-3 accuracy: {results[2]:.4%}")

In [ ]:
model.save("tsl_model.h5")

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.target_spec.supported_ops = 
    tf.lite.OpsSet.TFLITE_BUILTINS,    
    tf.lite.OpsSet.SELECT_TF_OPS       
    
converter._experimental_lower_tensor_list_ops = False

tflite_model = converter.convert()
with open("tsl_model.tflite", "wb") as f:
    f.write(tflite_model)

print("Success")


In [ ]:
import mediapipe as mp
from collections import deque
from tensorflow.keras.models import load_model

model     = "tsl_model.h5"
T              = 20
size       = 224
min_confidence = 0.7

class_names = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'behi', 'belhak', 'berda', 'beshwaya', 'esmi', 'express', 'fayek', 'hayej', 'ksir', 'masdoum', 'metghashesh', 'njewbek', 'omi', 'raked', 'tjewbni', 'twil', 'yaayshek', 'yakra', 'yekhdem', 'yemshi', 'Z', 'ahad', 'asam', 'baba', 'chorba', 'cv', 'erbaa', 'essalem', 'fhemt', 'jomaa', 'kafteji', 'khayeb', 'khmis', 'khouya', 'kosksi', 'lablebi', 'merci', 'okhti', 'sebt', 'skhouna', 'slata_meshwiya', 'thleth', 'thnin', 'tounsi']
classes_number = len(class_names)

model = load_model(model)
assert model.output_shape[-1] == classes_number, (
    f"Le modèle prévoit {model.output_shape[-1]} classes, "
    f"tu en as défini {classes_number} dans class_names."
)

mediapipe_hands = mp.solutions.hands
hands = mediapipe_hands.Hands(static_image_mode = False, number_of_hands = number_of_hands, detection_confidence = detection_confidence, tracking_confidence = tracking_confidence)

def box(landmarks, img_shape, margin=0.5):
    h, w, _ = img_shape
    xs = [lm.x for lm in landmarks.landmark]
    ys = [lm.y for lm in landmarks.landmark]
    min_x, max_x = max(min(xs), 0), min(max(xs), 1)
    min_y, max_y = max(min(ys), 0), min(max(ys), 1)
    x1, y1 = int(min_x * w), int(min_y * h)
    x2, y2 = int(max_x * w), int(max_y * h)
    dx, dy = int((x2 - x1) * margin), int((y2 - y1) * margin)
    return max(0, x1 - dx), max(0, y1 - dy), min(w, x2 + dx), min(h, y2 + dy)

cap = cv2.VideoCapture(0)

# Un seul deque partagé pour toutes les mains
frames_deque = deque(maxlen=T)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb)

    if results.multi_hand_landmarks:
        for lm in results.multi_hand_landmarks:
            x1, y1, x2, y2 = box(lm, frame.shape)

            roi = frame[y1:y2, x1:x2]
            if roi.size == 0:
                roi = frame
            roi = cv2.resize(roi, (size, size))
            roi_norm = roi.astype(np.float32) / 255.0

            # Ajoute l'image au buffer
            frames_deque.append(roi_norm)
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

            # Prédiction quand T frames sont collectées
            if len(frames_deque) == T:
                seq = np.stack(frames_deque, axis=0)
                seq = np.expand_dims(seq, axis=0)

                preds = model.predict(seq, verbose=0)[0]
                idx = np.argmax(preds)
                label = class_names[idx]
                prob = preds[idx]

                # Affichage
                text = f"{label} ({prob*100:.1f}%)"
                cv2.putText(frame, text, (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255,255,255), 2)

    cv2.imshow("Detection", frame)
    if cv2.waitKey(1) & 0xFF == 27:  # ESC pour quitter
        break

cap.release()
cv2.destroyAllWindows()
